In [1]:
import cartopy.crs as ccrs
import numpy as np
import xarray as xr
import pyremo as pr
import cmaps
import glob
from xstatstests import mannwhitneyu
from REMO2020_mod import season_mean, open_mfdataset, domainclass, plotclass, remo_plotter_2mods
import os
os.environ['PATH'] = '/work/ch0636/g300096/Python/envs/kernelR2020TEB/bin:' + os.environ['PATH']

/work/ch0636/g300096/Python/envs/kernelR2020TEB/lib/python3.10/site-packages/pyremo/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
ERROR 1: PROJ: proj_create_from_database: Open of /work/ch0636/g300096/Python/envs/kernelR2020TEB/share/proj failed


In [2]:
from dask.distributed import Client, progress
import dask
dask.config.set({'temporary_directory': '/scratch/g/g300096/dask/tmp'})
client = Client()
client

/work/ch0636/g300096/Python/envs/kernelR2020TEB/lib/python3.10/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 37545 instead
  warnings.warn(
/work/ch0636/g300096/Python/envs/kernelR2020TEB/lib/python3.10/contextlib.py:142: UserWarning: Creating scratch directories is taking a surprisingly long time. (4.69s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/g300096/levante-spawner-preset//proxy/37545/status,
Dashboard: /user/g300096/levante-spawner-preset//proxy/37545/status,Workers: 16
Total threads: 256,Total memory: 235.37 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35319,Workers: 0
Dashboard: /user/g300096/levante-spawner-preset//proxy/37545/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46435,Total threads: 16
Dashboard: /user/g300096/levante-spawner-preset//proxy/40901/status,Memory: 14.71 GiB
Nanny: tcp://127.0.0.1:43309,


In [22]:
# create variable info class for model
class modelclass:
    def __init__(self, userexp, runame, vegrun, urbvarname=''):
        self.userexp = userexp # XXXYYY
        self.runame = runame # name of the run
        self.vegrun = vegrun
        self.urbvarname = urbvarname

In [23]:
# variable to be plotted (T2MAX and T2MIN supported)
varplot="T2MIN"

In [24]:
# define the runs to be plotted
modelruns =[] # init

# no imove
modelruns.append(modelclass("036045","REMO2020 Veg",True))
modelruns.append(modelclass("036041","Rock-Veg",False,urbvarname="GLCC"))
modelruns.append(modelclass("036043","TEB-Veg",False,urbvarname="ECOSG_LCZ"))

#with imove
modelruns.append(modelclass("036046","REMO2020 iMOVE Veg",True))
modelruns.append(modelclass("036042","Rock-Veg iMOVE",False,urbvarname="LANDMATE"))
modelruns.append(modelclass("036044","TEB-Veg iMOVE",False,urbvarname="LANDMATE"))

In [25]:
# give path, prefix and suffix (the main name for the plots)
figpath = './figures/'
if(varplot == "T2MAX"):
    prefix = 'vegdiff_tempmax_selseas_'
    # or give absolute figure name (leave otherwise empty (''))
    afigname = 'fig11'
elif(varplot == "T2MIN"):
    prefix = 'vegdiff_tempmin_selseas_'
    # or give absolute figure name (leave otherwise empty (''))
    afigname = 'fig10'
else:
    print("varplot: "+varplot+" not correctly set (T2MIN or T2MAX)")
#
suffix = 'REMO2020_mainimove'

In [26]:
# REMO and EOBS datapath
rdata = './data_in/model_data/'
edata = './data_in/E-OBS/'
# urban fractions
remourbfile = rdata+"REMO_static/urban_fractions_EUR-11.nc"
# land-sea mask
remoblafile = rdata+"REMO_static/BLA_EUR-11.nc"

In [27]:
# open land-sea-mask
mask_lsm = xr.open_dataset(remoblafile).BLA.rename("mask_lsm").compute()

In [28]:
# define star year and end year
ystart=2011
yend=2020

In [29]:
# Define the order of seasons to be used (also when plotting)
seasons = ["DJF","JJA"]

In [30]:
yearlist = np.arange(ystart,yend+1)
for obj in modelruns:
    obj.filenames = []
    for year in yearlist:
        pattern = rdata+obj.userexp+"/results/"+str(year)+"/xm/e"+obj.userexp+"m*.nc"
        obj.filenames += glob.glob(pattern)
    obj.filenames.sort()

In [31]:
# Set halo zone length (will be cutted out)
halo = 8

In [32]:
# shall we filter the results using a statistical test
lfilterstat=True

In [33]:
# Limit for urban/non-urban (greater or equal urban fraction -> urban, and vice versa)
urlim = 0.1

In [34]:
# Student's t-test threshold
t_test_lim = 0.05

In [35]:
# land fraction cut-off limit (land fraction equal or below -> excluded)
cland = 0.0

In [36]:
# calculate the T2m difference to veg
for obj in modelruns:
    ds = open_mfdataset(obj.filenames, parallel=False, chunks='auto')
    t2m_tmp = pr.parse_dates(ds[varplot]) # has the 2-meter dimension and its name might change. Luckily it is in position 1 and we can remove it
    t2m_tmp = t2m_tmp.squeeze(dim=t2m_tmp.dims[1],drop=True)
    # Merge data (get identical coordinates)
    t2m_all = xr.merge([t2m_tmp, mask_lsm], compat="override", join="override")
    # use only if 50% or more is land
    t2m = t2m_all[varplot].where(t2m_all.mask_lsm > cland,drop=False).compute()
    # Calculate absolute values
    if(obj.vegrun):
        abs_vals = t2m.compute()
        abs_vals_mean = season_mean(abs_vals)
        abs_vals_mean = abs_vals_mean.isel(rlat=slice(halo+1,abs_vals_mean.sizes["rlat"]-halo+1), rlon=slice(halo+1,abs_vals_mean.sizes["rlon"]-halo+1)).compute()
        obj.vardiff = abs_vals_mean # to be used later
    else:
        # get identical coordinates
        final_tmp = xr.merge([t2m,abs_vals.rename("abs_vals")], compat="override", join="override").compute()
        obj.vardiff = season_mean(final_tmp[varplot] - final_tmp.abs_vals).squeeze()
        # calculate and filter statistics, if active
        if(lfilterstat):
            # loop over seasons and calculate statistics
            ii = 0
            for seas in seasons:
                final_tmp_seas = final_tmp.sel(time=final_tmp['time.season'] == seas)
                res_tmp = (mannwhitneyu(final_tmp_seas[[varplot]], final_tmp_seas[["abs_vals"]].rename({"abs_vals":varplot}),dim="time")).expand_dims({"season": [ seas ]}).compute()
                if(ii == 0):
                    result_ind = res_tmp
                else:
                    result_ind = xr.concat([result_ind, res_tmp],dim="season").compute()
                # increase counter
                ii = ii + 1
                # delete tmp
                del final_tmp_seas, res_tmp
            # filter 
            obj.vardiff = obj.vardiff.where(result_ind.pvalue < t_test_lim, drop=False)
            # can be also used to mask urban areas, e.g. >= 0.1 or < 0.1
            urbfr = (xr.open_dataset(remourbfile)[obj.urbvarname]).isel(rlat=slice(halo+1,obj.vardiff.sizes["rlat"]-halo+1), rlon=slice(halo+1,obj.vardiff.sizes["rlon"]-halo+1)).compute()
            #obj.vardiff = obj.vardiff.where(urbfr < urlim, drop=False).squeeze()
        obj.vardiff = obj.vardiff.isel(rlat=slice(halo+1,obj.vardiff.sizes["rlat"]-halo+1), rlon=slice(halo+1,obj.vardiff.sizes["rlon"]-halo+1)).compute()
        #ones_2d = xr.ones_like(urbfr)
        # get area weights
        weight = np.cos(np.deg2rad(urbfr.rlat))
        for seas in seasons:
            nourb = (t2m_all.mask_lsm.isel(rlat=slice(halo+1,obj.vardiff.sizes["rlat"]-halo+1), rlon=slice(halo+1,obj.vardiff.sizes["rlon"]-halo+1))-urbfr).where(result_ind.sel(season=seas).pvalue < t_test_lim, drop=False).where(urbfr < urlim).weighted(weight).sum(dim=("rlat", "rlon")).values
            urb = urbfr.where(urbfr >= urlim).weighted(weight).sum(dim=("rlat", "rlon")).values
            print(obj.runame,seas,urb,nourb,(nourb+urb)/urb)
        # not supported, set to zero
        obj.mask = np.zeros(len(seasons))
    #
    pole = (ds.rotated_latitude_longitude.grid_north_pole_longitude,ds.rotated_latitude_longitude.grid_north_pole_latitude)
    rotated_pole = ccrs.RotatedPole(pole[0], pole[1])
    extent = ([ds.rlon.min(), ds.rlon.max(), ds.rlat.min(), ds.rlat.max()], rotated_pole)
    # clean memory
    del ds, t2m, t2m_tmp, t2m_all
# clean memory
del abs_vals, final_tmp

Rock-Veg DJF [242.31651223] [0.] [1.]
Rock-Veg JJA [242.31651223] [3.66692106] [1.01513277]
TEB-Veg DJF [630.71426652] [0.87265189] [1.00138359]
TEB-Veg JJA [630.71426652] [5.1615507] [1.00818366]
Rock-Veg iMOVE DJF [743.24131242] [0.] [1.]
Rock-Veg iMOVE JJA [743.24131242] [0.] [1.]
TEB-Veg iMOVE DJF [743.24131242] [4.85295711] [1.00652945]
TEB-Veg iMOVE JJA [743.24131242] [13.62259574] [1.01832863]


In [37]:
# define the domains to be plotted
domaininfo =[] # init
domaininfo.append(domainclass("EU",[-22.5,47.5,30,72.5], hspace=0.35))

In [38]:
# define plotclasses for absolute and difference (model) plots
if(varplot == "T2MAX"):
    plabela = 'T$_{\mathrm{max}}$ [K]'
    plabeld = '$\Delta$T$_{\mathrm{max}}$ [K]'
    vmin = 265
    vmax = 305
elif(varplot == "T2MIN"):
    plabela = 'T$_{\mathrm{min}}$ [K]'
    plabeld = '$\Delta$T$_{\mathrm{min}}$ [K]'
    vmin = 255
    vmax = 295
absplotc = plotclass(label=plabela, lspace=np.linspace(0,1,32), vmin=vmin, vmax=vmax, linvert=False, cmapp=cmaps.BlAqGrYeOrReVi200)
modplotc = plotclass(label=plabeld, lspace=np.linspace(0,1,30), vmin=-1.5, vmax=1.5, linvert=False, cmapp=cmaps.BlueYellowRed)

In [39]:
# do some separation
modelruns_1 = modelruns[1:3] # first one not really used for diffs
modelruns_2 = modelruns[4:]

In [40]:
# Loop over domains
for dom in range(len(domaininfo)):
    # Create figure name
    figname=prefix+domaininfo[dom].name+'_'+suffix
    # check if we want to use absolute figure name
    if(afigname):
        figname = afigname
    # Call plotter
    remo_plotter_2mods(absplotc,modplotc,domaininfo[dom],modelruns_1,modelruns_2,modelruns[0].runame,modelruns[0].vardiff,modelruns[3].runame,modelruns[3].vardiff,seasons,pole,False,figpath,figname)